In [1]:
import pandas as pd
import sys
sys.path.insert(0, "..")                # sys module to look for utils.py on project's root
from utils import fill_empty_rows, import_datasets_to_db, create_file_path

In [2]:
# Read the Chrysler .csv file
df_chrysler = pd.read_csv(create_file_path("chrysler.csv"))

In [3]:
# Read Dodge Cummins .csv file
df_cummins = pd.read_csv(create_file_path("dodge-cumminsR.csv"))

In [8]:
# Check the cols available
df_chrysler.columns

Index(['Contlr', 'ModuleType SBECIII', 'Generic Hardware P/N',
       'Latest Software', 'Older Software P/N Versions', 'Year', 'Body',
       'Engine', 'Software Description', 'Trans Type',
       'Sales Codes\neng trans emissions', 'VIN Example'],
      dtype='object')

In [9]:
# Get the index with empty rows
df_chrysler_empty_rows_index = df_chrysler[df_chrysler["Contlr"].isna() == True].index

In [10]:
# Drop the empty rows
df_chrysler_drop_empty = df_chrysler.drop(df_chrysler_empty_rows_index)

In [11]:
# Drop the repeated columns
df_chrysler_repeated_cols_index = df_chrysler_drop_empty[df_chrysler_drop_empty["Contlr"] == "Contlr"].index

In [12]:
# Drop the repeated cols
df_chrysler_drop_empty = df_chrysler_drop_empty.drop(df_chrysler_repeated_cols_index)

In [13]:
# Check the columns of the df
df_chrysler_drop_empty.columns

Index(['Contlr', 'ModuleType SBECIII', 'Generic Hardware P/N',
       'Latest Software', 'Older Software P/N Versions', 'Year', 'Body',
       'Engine', 'Software Description', 'Trans Type',
       'Sales Codes\neng trans emissions', 'VIN Example'],
      dtype='object')

# Cols to keep on df Chrysler:
- Hardware
- Latest Software
- Older Software P/N Versions

In [14]:
# List with cols to keep
cols_to_keep = ["Generic Hardware P/N", "Latest Software", "Older Software P/N Versions"]

In [15]:
# Create a df with only the cols to keep
df_chrysler_w_cols_to_keep = df_chrysler_drop_empty[cols_to_keep]

In [16]:
# Map the columns names will remain
cols_ren_map = {
    "Generic Hardware P/N": "hardware",
    "Latest Software": "pn1",
    "Older Software P/N Versions": "pn2"
}

In [17]:
# Rename the columns
df_renamed_cols = df_chrysler_w_cols_to_keep.rename(columns=cols_ren_map)

In [18]:
# Confirm if the columns were renamed
df_renamed_cols.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8353 entries, 0 to 8416
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  8110 non-null   object
 1   pn1       8319 non-null   object
 2   pn2       8039 non-null   object
dtypes: object(3)
memory usage: 261.0+ KB


In [19]:
# Call the function to fill out null rows
df_chrysler_fill_nullValues = fill_empty_rows(df_renamed_cols)

In [20]:
# Show the cols
df_chrysler_fill_nullValues.columns

Index(['hardware', 'pn1', 'pn2'], dtype='object')

In [21]:
# Add 3 columns to concatenate the Dodge Cummins Dataframe
df_chrysler_added_cols = df_chrysler_fill_nullValues.assign(
    pn3=None,
    pn4=None,
    pn5=None
)

In [22]:
# Call the function to fill out rows with None values
df_chrysler_fill_nulls = fill_empty_rows(df_chrysler_added_cols)

In [23]:
# Check Chrysler df
df_chrysler_fill_nulls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8353 entries, 0 to 8416
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   hardware  8353 non-null   object
 1   pn1       8353 non-null   object
 2   pn2       8353 non-null   object
 3   pn3       8353 non-null   object
 4   pn4       8353 non-null   object
 5   pn5       8353 non-null   object
dtypes: object(6)
memory usage: 456.8+ KB


In [24]:
# Show head
df_chrysler_head = df_chrysler_fill_nulls.head(5)

### **Dodge Cummins Dataset Cleaning**

### Cols to keep:
- ModTYPE
- Hardware
- CUMMINS P/N
- CUMMINS E/C
- Latest Software
- Older Software Versions

In [31]:
# Map all cols to keep
cummins_cols_keep_map = {
    "ModTYPE": "pn1",
    "Hardware": "hardware",
    "CUMMINS P/N": "pn2",
    "CUMMINS E/C": "pn3",
    "Latest Software": "pn4",
    "Older Software Versions": "pn5"     
}

In [32]:
# Rename the cols
df_cummins_rn_cols = df_cummins.rename(columns=cummins_cols_keep_map)

In [33]:
# Check the columns
df_cummins_rn_cols.columns

Index(['pn1', 'hardware', 'pn2', 'pn3', 'pn4', 'pn5', 'Model Year and Trans',
       'Engine', 'Description', 'Example VINs'],
      dtype='object')

In [34]:
# Reorder the columns and keep only columns needed
df_cummins_reorder_cols = df_cummins_rn_cols[['hardware', 'pn1', 'pn2', 'pn3', 'pn4', 'pn5']]

In [35]:
# Call the function to fill out rows with None values
df_cummins_fill_nulls = fill_empty_rows(df_cummins_reorder_cols)

e:\Language_Projects\Language_Projects\Python\Flagship_1\part-number-datasets-cleaning\notebooks\..\utils.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [36]:
# Show the head to confirm the changes
df_cummins_head = df_cummins_reorder_cols.head(4)

In [37]:
# Concate both dataframes to create only 1
df_chrysler_final = pd.concat([df_chrysler_fill_nulls, df_cummins_reorder_cols], sort=False)

In [38]:
# Reset index
df_chrysler_reset_index = df_chrysler_final.reset_index(drop=True)

In [39]:
# Call the function to import the df cleaned to the postgreSQLdb
import_datasets_to_db(df_renamed_cols, 'part_numbers_chrysler')

353